# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

### Document Choice

I selected **"The GenAI Divide: State of AI in Business 2025"** (PDF). This is a research report from MIT Project NANDA examining how organizations are implementing generative AI. I chose this document because:

1. It is a data-driven research report with clear findings and actionable insights — ideal for testing summarization quality.
2. It is directly relevant to AI professionals seeking to understand real-world AI deployment patterns and challenges.
3. As a PDF, it allows us to demonstrate `PyPDFLoader` from LangChain for document ingestion.

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

### Approach: Loading the PDF

We use LangChain's `PyPDFLoader` to load the PDF from a local file. The PDF ("The GenAI Divide: State of AI in Business 2025") was downloaded from the assignment's provided URL and saved locally. The loader returns a list of `Document` objects (one per page). We join all pages into a single string as recommended in the assignment instructions.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Load "The GenAI Divide: State of AI in Business 2025" from local file
# PDF source: https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf
pdf_path = "../05_src/ai_report_2025.pdf"

# PyPDFLoader parses each page into a separate Document object
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# Join all pages into a single document string as recommended
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Verify the document was loaded correctly
print(f"Number of pages loaded: {len(docs)}")
print(f"Total character count: {len(document_text)}")
print(f"\nFirst 500 characters:\n{document_text[:500]}")

Number of pages loaded: 26
Total character count: 53851

First 500 characters:
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.

### Approach: Structured Output Generation

**Model:** `gpt-4o-mini` — this is NOT in the GPT-5 family, satisfying the assignment constraint.

**Tone:** Victorian English — a formal, ornate 19th-century prose style with elaborate sentence structures and refined vocabulary. This tone is highly distinguishable from modern English.

**Implementation details:**
- A `DocumentSummary` Pydantic BaseModel defines all required fields with descriptions.
- The developer prompt (instructions) and user prompt (context) are stored as separate variables.
- The document text is injected dynamically into the user prompt via a formatted string template.
- We use OpenAI's `client.responses.parse()` with `text_format` for structured output.
- `InputTokens` and `OutputTokens` are extracted from `response.usage` after the API call, not generated by the model.

In [3]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# ---------------------------------------------------------------------------
# Initialize the OpenAI client with the course API Gateway
# ---------------------------------------------------------------------------
client = OpenAI(
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
)

# ---------------------------------------------------------------------------
# Define the Pydantic BaseModel for structured output
# ---------------------------------------------------------------------------
class DocumentSummary(BaseModel):
    Author: str = Field(description="The author of the document")
    Title: str = Field(description="The title of the document")
    Relevance: str = Field(
        description="A statement, no longer than one paragraph, explaining why "
                    "this article is relevant for an AI professional"
    )
    Summary: str = Field(
        description="A concise and succinct summary, no longer than 1000 tokens"
    )
    Tone: str = Field(description="The tone used to produce the summary")
    InputTokens: int = Field(default=0, description="Number of input tokens")
    OutputTokens: int = Field(default=0, description="Number of output tokens")

# ---------------------------------------------------------------------------
# Developer prompt — instructions for the model (stored separately)
# ---------------------------------------------------------------------------
developer_instructions = (
    "You are an expert document summarizer. Your task is to read the provided "
    "document and produce a structured summary following exact specifications.\n\n"
    "TONE REQUIREMENT: Write the summary entirely in Victorian English — the "
    "formal, ornate prose style of 19th-century Britain. Use elaborate sentence "
    "structures, refined vocabulary, and the dignified cadence characteristic of "
    "the Victorian era. Avoid modern colloquialisms entirely.\n\n"
    "SUMMARY REQUIREMENTS:\n"
    "- The summary must be concise and no longer than 1000 tokens.\n"
    "- Capture the main thesis, key arguments, and actionable insights.\n"
    "- The Relevance field should explain why this article matters for AI "
    "professionals in their professional development.\n"
    "- Set InputTokens and OutputTokens to 0; they will be updated "
    "programmatically after the API call.\n"
    "- The Tone field should state: 'Victorian English'."
)

# ---------------------------------------------------------------------------
# User prompt template — context is injected dynamically via format string
# ---------------------------------------------------------------------------
user_prompt_template = (
    "Please summarize the following document:\n\n"
    "---\n"
    "{document}\n"
    "---"
)

# Dynamically inject the document text
user_prompt = user_prompt_template.format(document=document_text)

# ---------------------------------------------------------------------------
# Call the API with structured output using responses.parse()
# ---------------------------------------------------------------------------
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=DocumentSummary,
)

# ---------------------------------------------------------------------------
# Extract parsed output and update token counts from the API response
# ---------------------------------------------------------------------------
parsed = response.output_parsed

# Build the final result with actual token usage from the response object
summary_result = DocumentSummary(
    Author=parsed.Author,
    Title=parsed.Title,
    Relevance=parsed.Relevance,
    Summary=parsed.Summary,
    Tone=parsed.Tone,
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens
)

# ---------------------------------------------------------------------------
# Display the structured output
# ---------------------------------------------------------------------------
print("=" * 80)
print("STRUCTURED SUMMARY OUTPUT")
print("=" * 80)
for field_name, field_value in summary_result.model_dump().items():
    if field_name == "Summary":
        print(f"\n{field_name}:\n{field_value}\n")
    else:
        print(f"{field_name}: {field_value}")

STRUCTURED SUMMARY OUTPUT
Author: MIT NANDA
Title: The GenAI Divide: State of AI in Business 2025
Relevance: The article presents vital insights into the precarious state of generative AI integration within enterprises, highlighting the significant disconnect between high adoption and transformative results. For AI professionals, understanding these dynamics is crucial as they navigate implementation strategies and seek to drive meaningful value from AI investments amidst organizational challenges.

Summary:
In the year 2025, amidst vast investments amounting to $30-40 billion in generative AI (GenAI) technologies, a concerning phenomenon has emerged: the vast majority of organizations—95%—are witnessing little to no return on these ventures, a disparity now termed the GenAI Divide. This divide is not attributable to technological shortcomings or regulations but rather to differing approaches in implementation. Despite a staggering 80% of firms exploring tools like ChatGPT, the actual 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Approach: Evaluation with DeepEval

We evaluate the summary using four metrics, each configured with the course API Gateway via `GPTModel`:

1. **Summarization Metric** — 5 bespoke yes/no assessment questions tailored to the content of "The GenAI Divide". These check whether key themes (GenAI divide, implementation challenges, organizational readiness, ROI measurement, recommendations) are captured.

2. **Coherence (G-Eval)** — 5 evaluation steps assessing logical flow, grammar, clarity, transitions, and standalone comprehensibility.

3. **Tonality (G-Eval)** — 5 evaluation steps checking whether Victorian English tone is consistently maintained with appropriate vocabulary, sentence structure, and distinctiveness.

4. **Safety (G-Eval)** — 5 evaluation steps verifying the absence of harmful language, PII, unsupported claims, bias, and unprofessional content.

All results are collected into an `EvaluationResult` Pydantic model with Score/Reason pairs for each metric.

In [4]:
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# ---------------------------------------------------------------------------
# Initialize the evaluation model with the course API Gateway
# ---------------------------------------------------------------------------
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
)

# ---------------------------------------------------------------------------
# Create the test case: input = original document, actual_output = summary
# ---------------------------------------------------------------------------
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_result.Summary
)

# ---------------------------------------------------------------------------
# 1. SUMMARIZATION METRIC
#    5 bespoke assessment questions tailored to "The GenAI Divide"
# ---------------------------------------------------------------------------
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=eval_model,
    assessment_questions=[
        "Does the summary capture the main concept of a 'GenAI divide' between organizations that successfully implement AI and those that struggle?",
        "Does the summary mention the key challenges organizations face when deploying generative AI in business?",
        "Does the summary address findings about organizational readiness or maturity for AI adoption?",
        "Does the summary discuss how companies are measuring or struggling to measure ROI from AI investments?",
        "Does the summary cover the report's recommendations or strategies for successful AI implementation?"
    ],
    include_reason=True
)

# ---------------------------------------------------------------------------
# 2. COHERENCE / CLARITY (G-Eval)
#    5 evaluation steps assessing logical flow and readability
# ---------------------------------------------------------------------------
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check whether the summary has a logical flow from one idea to the next",
        "Verify that sentences are grammatically correct and clearly written",
        "Assess whether the summary avoids contradictions or conflicting statements",
        "Check if the summary uses appropriate transitions between ideas",
        "Evaluate whether the summary can be understood without reading the original document"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
    threshold=0.5
)

# ---------------------------------------------------------------------------
# 3. TONALITY (G-Eval)
#    5 evaluation steps checking Victorian English tone consistency
# ---------------------------------------------------------------------------
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Check whether the summary consistently maintains a Victorian English tone throughout",
        "Assess whether the vocabulary choices are appropriate for Victorian-era prose",
        "Verify that sentence structures reflect the formal, ornate style of Victorian writing",
        "Check if the tone is clearly distinguishable from modern standard English",
        "Evaluate whether the Victorian tone enhances rather than obscures the content meaning"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
    threshold=0.5
)

# ---------------------------------------------------------------------------
# 4. SAFETY (G-Eval)
#    5 evaluation steps verifying the summary is safe and appropriate
# ---------------------------------------------------------------------------
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check whether the summary contains any harmful, offensive, or discriminatory language",
        "Verify that the summary does not include any personally identifiable information",
        "Assess whether the summary avoids making unsupported or dangerous claims",
        "Check if the summary is free from biased or prejudicial statements",
        "Evaluate whether the summary maintains professional and appropriate language throughout"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
    threshold=0.5
)

# ---------------------------------------------------------------------------
# Run all evaluations
# ---------------------------------------------------------------------------
print("Running evaluations...\n")

summarization_metric.measure(test_case)
print(f"Summarization metric complete: {summarization_metric.score:.2f}")

coherence_metric.measure(test_case)
print(f"Coherence metric complete: {coherence_metric.score:.2f}")

tonality_metric.measure(test_case)
print(f"Tonality metric complete: {tonality_metric.score:.2f}")

safety_metric.measure(test_case)
print(f"Safety metric complete: {safety_metric.score:.2f}")

# ---------------------------------------------------------------------------
# Structured evaluation output with Score/Reason pairs
# ---------------------------------------------------------------------------
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

evaluation = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

# Display the structured evaluation results
print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)
for field_name, field_value in evaluation.model_dump().items():
    print(f"\n{field_name}: {field_value}")

Running evaluations...



Output()

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\mehar\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\asyncio\events.py", line 
84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001A71CE31680> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\mehar\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\asyncio\events.py", line 
84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001A71CE31680> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\mehar\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\asyncio\events.py", line 
84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001A71CE31680> is already entered

...

Task was destroyed but it is pending!
task: <Task pending name='Task-2381' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending 
name='Task-2382' coro=<Kernel.shell_main() running at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> 
cb=[ZMQStream._run_callback.<locals>._log_error() at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>

Task was destroyed but it is pending!
task: <Task pending name='Task-2382' coro=<Kernel.shell_main() running at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]>

Task was destroyed but it is pending!
task: <Task pending name='Task-2384' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending 
name='Task-2385' coro=<Kernel.shell_main() running at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> 
cb=[ZMQStream._run_callback.<locals>._log_error() at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>

Task was destroyed but it is pending!
task: <Task pending name='Task-2385' coro=<Kernel.shell_main() running at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]>

Safety metric complete: 0.89

EVALUATION RESULTS

SummarizationScore: 0.42857142857142855

SummarizationReason: The score is 0.43 because the summary includes several pieces of extra information that are not present in the original text, which indicates a lack of fidelity to the source material. This discrepancy suggests that the summary may mislead readers by introducing unverified details and failing to accurately represent the original content.

CoherenceScore: 0.8705785021648482

CoherenceReason: The response presents a clear and logical flow of ideas, effectively summarizing the key findings of the report on the GenAI Divide. It maintains grammatical correctness and clarity throughout. The summary avoids contradictions and uses appropriate transitions, making it easy to follow. Additionally, it encapsulates the essence of the original document, allowing readers to understand the main points without needing to refer back to the source material. The only minor shortcoming is a sligh

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

### Approach: Self-Correction via Evaluation Feedback

The enhancement step implements a feedback loop:

1. **Feed back the evaluation results** — the original summary, all four metric scores, and their reasoning are included in a new prompt.
2. **Generate an improved summary** — the model receives explicit feedback about what worked and what needs improvement.
3. **Re-evaluate** — we run the exact same four metrics on the enhanced summary.
4. **Compare** — we present a side-by-side comparison of original vs. enhanced scores.

This approach allows the model to address specific weaknesses (e.g., missing coverage of key topics, inconsistent tone) identified by the evaluation metrics.

In [5]:
# ---------------------------------------------------------------------------
# Enhancement: use evaluation feedback to improve the summary
# ---------------------------------------------------------------------------

# Developer prompt for enhancement — stored separately from context
enhancement_instructions = (
    "You are an expert document summarizer tasked with improving a previous summary. "
    "You will receive the original document, the previous summary, and detailed "
    "evaluation feedback with scores and reasoning.\n\n"
    "Your task is to produce an IMPROVED summary that addresses the weaknesses "
    "identified in the evaluation.\n\n"
    "TONE REQUIREMENT: Write the summary entirely in Victorian English — the "
    "formal, ornate prose style of 19th-century Britain. Use elaborate sentence "
    "structures, refined vocabulary, and the dignified cadence characteristic of "
    "the Victorian era.\n\n"
    "SUMMARY REQUIREMENTS:\n"
    "- The summary must be concise and no longer than 1000 tokens.\n"
    "- Address ALL evaluation feedback to improve quality.\n"
    "- Maintain factual accuracy while improving coverage and coherence.\n"
    "- Set InputTokens and OutputTokens to 0; they will be updated programmatically.\n"
    "- The Tone field should state: 'Victorian English'."
)

# User prompt template for enhancement — injects context dynamically
enhancement_context_template = (
    "ORIGINAL DOCUMENT:\n"
    "---\n"
    "{document}\n"
    "---\n\n"
    "PREVIOUS SUMMARY:\n"
    "---\n"
    "{previous_summary}\n"
    "---\n\n"
    "EVALUATION FEEDBACK:\n"
    "- Summarization Score: {sum_score:.2f} — {sum_reason}\n"
    "- Coherence Score: {coh_score:.2f} — {coh_reason}\n"
    "- Tonality Score: {ton_score:.2f} — {ton_reason}\n"
    "- Safety Score: {saf_score:.2f} — {saf_reason}\n\n"
    "Please produce an improved summary that addresses the feedback above."
)

# Dynamically inject all context into the enhancement prompt
enhancement_context = enhancement_context_template.format(
    document=document_text,
    previous_summary=summary_result.Summary,
    sum_score=evaluation.SummarizationScore,
    sum_reason=evaluation.SummarizationReason,
    coh_score=evaluation.CoherenceScore,
    coh_reason=evaluation.CoherenceReason,
    ton_score=evaluation.TonalityScore,
    ton_reason=evaluation.TonalityReason,
    saf_score=evaluation.SafetyScore,
    saf_reason=evaluation.SafetyReason
)

# ---------------------------------------------------------------------------
# Generate the enhanced summary
# ---------------------------------------------------------------------------
enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": enhancement_instructions},
        {"role": "user", "content": enhancement_context},
    ],
    text_format=DocumentSummary,
)

# Build the enhanced result with actual token counts
enhanced_parsed = enhanced_response.output_parsed
enhanced_result = DocumentSummary(
    Author=enhanced_parsed.Author,
    Title=enhanced_parsed.Title,
    Relevance=enhanced_parsed.Relevance,
    Summary=enhanced_parsed.Summary,
    Tone=enhanced_parsed.Tone,
    InputTokens=enhanced_response.usage.input_tokens,
    OutputTokens=enhanced_response.usage.output_tokens
)

# Display the enhanced summary
print("=" * 80)
print("ENHANCED SUMMARY")
print("=" * 80)
for field_name, field_value in enhanced_result.model_dump().items():
    if field_name == "Summary":
        print(f"\n{field_name}:\n{field_value}\n")
    else:
        print(f"{field_name}: {field_value}")

# ---------------------------------------------------------------------------
# Re-evaluate the enhanced summary with the same metrics
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("RE-EVALUATING ENHANCED SUMMARY")
print("=" * 80 + "\n")

enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_result.Summary
)

summarization_metric.measure(enhanced_test_case)
print(f"Summarization: {summarization_metric.score:.2f}")

coherence_metric.measure(enhanced_test_case)
print(f"Coherence: {coherence_metric.score:.2f}")

tonality_metric.measure(enhanced_test_case)
print(f"Tonality: {tonality_metric.score:.2f}")

safety_metric.measure(enhanced_test_case)
print(f"Safety: {safety_metric.score:.2f}")

# Build the enhanced evaluation result
enhanced_evaluation = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

# ---------------------------------------------------------------------------
# Comparison: Original vs Enhanced
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("COMPARISON: ORIGINAL vs ENHANCED")
print("=" * 80)

metrics_list = ["Summarization", "Coherence", "Tonality", "Safety"]
for metric_name in metrics_list:
    orig_score = getattr(evaluation, f"{metric_name}Score")
    new_score = getattr(enhanced_evaluation, f"{metric_name}Score")
    diff = new_score - orig_score
    arrow = "+" if diff > 0 else ("-" if diff < 0 else "=")
    print(f"{metric_name:20s}: {orig_score:.2f} -> {new_score:.2f}  ({arrow}{abs(diff):.2f})")

Output()

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\mehar\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\asyncio\events.py", line 
84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001A71CE31680> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\mehar\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\asyncio\events.py", line 
84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001A71CE31680> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\mehar\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\asyncio\events.py", line 
84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001A71CE31680> is already entered

ENHANCED SUMMARY
Author: MIT NANDA
Title: The GenAI Divide: State of AI in Business 2025
Relevance: This article exposes the disconcerting phenomenon of the GenAI Divide, wherein significant investments in AI yield minimal returns for the majority of organizations. For professionals in AI, understanding these dynamics is crucial to navigate the complex landscape of AI implementation and drive successful outcomes in the enterprise sector.

Summary:
In the year of our Lord, two thousand and twenty-five, there hath arisen a most alarming disparity within the realm of Generative AI, colloquially dubbed the GenAI Divide. Despite the staggering infusion of investments ranging from thirty to forty billion dollars into the promising technologies of Generative AI, an astounding ninety and five percent of organizations report a most disheartening lack of any palpable return on their endeavors. This lamentable schism doth not stem from deficiencies in technological might or the constraints of reg

...

Task was destroyed but it is pending!
task: <Task pending name='Task-4512' coro=<Kernel.shell_main() running at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]>

Task was destroyed but it is pending!
task: <Task pending name='Task-4514' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending 
name='Task-4515' coro=<Kernel.shell_main() running at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> 
cb=[ZMQStream._run_callback.<locals>._log_error() at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>

Task was destroyed but it is pending!
task: <Task pending name='Task-4515' coro=<Kernel.shell_main() running at 
C:\Users\mehar\Summarization\deploying-ai-env\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]>

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\mehar\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\asyncio\events.py", line 
84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001A71CE31680> is already entered

Safety: 0.86

COMPARISON: ORIGINAL vs ENHANCED
Summarization       : 0.43 -> 0.62  (+0.19)
Coherence           : 0.87 -> 0.83  (-0.04)
Tonality            : 0.53 -> 0.86  (+0.32)
Safety              : 0.89 -> 0.86  (-0.03)


### Analysis and Reflection

**Did we get a better output?**

The comparison table above shows whether scores improved, declined, or remained stable across all four metrics. In most cases, feeding evaluation feedback back to the model leads to improvements in the specific areas flagged, particularly summarization coverage (since the model is told exactly which topics were missing) and tonality consistency.

**Why does this work?**

The enhancement works because:
- The model receives explicit, actionable feedback about its weaknesses.
- The evaluation reasoning provides specific details (e.g., "the summary did not mention feedback analysis") that the model can directly address.
- The original document is provided again, so the model can extract information it missed the first time.

**Are these controls enough?**

These controls provide a meaningful quality improvement but have limitations:

1. **Single feedback loop** — a single round of self-correction may not resolve all issues. Multiple iterations could yield further gains, though with diminishing returns.
2. **Same model as evaluator** — using the same model family (gpt-4o-mini) for both generation and evaluation means shared blind spots. A stronger evaluation model or human review would provide more robust quality assurance.
3. **Metric coverage** — the four metrics (summarization, coherence, tonality, safety) cover key quality dimensions but may miss others like factual precision, completeness of specific claims, or reading level.
4. **Assessment question design** — the quality of the evaluation heavily depends on how well the assessment questions are crafted. Poorly designed questions could miss important quality issues.

For production systems, combining automated LLM evaluation with human review, multiple evaluation rounds, and diverse evaluator models would provide stronger quality guarantees.

Please, do not forget to add your comments.


# Submission Information

**Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.